# L7.6 · SmolVLA: las 10 pasadas

| Notebook | Qué abre |
|---|---|
| L7.1 | El mapa: qué entra, qué sale, dónde están los 450 M de parámetros |
| L7.2 | La imagen: de la foto a 64 tokens |
| L7.3 | La fila del VLM: palabras, estado y quién puede mirar a quién |
| L7.4 | Adentro del VLM: qué miran las palabras, por qué 16 de 32 capas, qué queda guardado |
| `06` (flujo) | Entrenarlo con nuestros datos |
| L7.5 | El experto de acciones: la brújula de L6 con ojos |
| **L7.6** | Las 10 pasadas: un bloque de verdad pasando de ruido a movimiento |
| L7.7 | El entrenamiento por dentro |
| `07` (flujo) | Evaluación contra ACT, en los mismos episodios |

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import time

import matplotlib.pyplot as plt
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR
from toolkit.policy import smolvla_internals as vla
from toolkit.policy.smolvla_g1_base import load_g1_policy

plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
FPS = 30

TRAINED = CHECKPOINTS_DIR / "smolvla_g1_v1" / "checkpoints" / "last" / "pretrained_model"
train_meta = LeRobotDataset("local/pick_mameluco_act_v1_train", root=DATASETS_DIR / "local" / "pick_mameluco_act_v1_train").meta
heldout = LeRobotDataset("local/pick_mameluco_act_v1_heldout", root=DATASETS_DIR / "local" / "pick_mameluco_act_v1_heldout",
                         delta_timestamps={"action": [i / FPS for i in range(50)]})
names = heldout.meta.features["action"]["names"]
joint_names = names[0] if isinstance(names[0], list) else names
policy, preprocess, postprocess = load_g1_policy(TRAINED, train_meta.stats, device)
model = policy.model

# Un instante de un episodio que el modelo nunca vio al entrenar: el de L7.5.
frame = heldout[int(heldout.meta.episodes["dataset_from_index"][1]) + 120]
obs = vla.prepare_observation(policy, preprocess(dict(frame)))
truth = frame["action"].numpy()
watch = [joint_names.index(n) for n in ["kRightShoulderPitch", "kRightElbow", "kRightWristPitch", "kRightHandThumb"]]


def to_radians(block):
    """Del bloque del modelo (normalizado, 32 números) a radianes de las 26 articulaciones."""
    return postprocess(block[..., :26]).cpu().numpy()

## 1. El problema

Ya tenemos la brújula: el experto (L7.5). Generar es caminar con ella desde el
ruido, de a pasitos, como en L6.3. La diferencia es que ahora el "punto" es un
bloque de 50 acciones × 26 articulaciones: no se puede dibujar entero, pero sí
mirar algunas articulaciones a lo largo de las 10 pasadas.

## 2. Un bloque, pasada por pasada

Sorteamos ruido y caminamos las 10 pasadas, guardando el bloque después de cada
una. Dibujamos cuatro articulaciones del brazo derecho y la mano: de claro
(ruido) a oscuro (terminado), y en negro lo que hizo la persona.

In [ ]:
torch.manual_seed(0)
noise = torch.randn(1, model.config.chunk_size, model.config.max_action_dim, device=device)
with torch.no_grad():
    path = vla.sample_with_path(model, obs, noise)
steps = [to_radians(x)[0] for x in path]  # 11 bloques: ruido + 10 pasadas

seconds = np.arange(50) / FPS
fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
shades = plt.cm.Reds(np.linspace(0.15, 1, len(steps)))
for ax, j in zip(axes, watch):
    for k, (block, shade) in enumerate(zip(steps, shades)):
        ax.plot(seconds, block[:, j], color=shade, lw=1.6 if k == len(steps) - 1 else 0.8)
    ax.plot(seconds, truth[:, j], color="black", ls="--", label="la persona")
    ax.set_title(joint_names[j], fontsize=9)
    ax.set_xlabel("s")
    ax.set_ylim(truth[:, j].min() - 0.6, truth[:, j].max() + 0.6)
axes[0].legend(fontsize=8)
plt.tight_layout()
print(f"error final contra la persona: {np.degrees(np.abs(steps[-1] - truth).mean()):.1f}° de promedio")

Las curvas claras son ruido: cada acción del bloque salta para cualquier
lado, sin relación con la de al lado. Pasada a pasada los saltos se achican y
la curva se acerca a una trayectoria suave.

Fijate que las curvas intermedias parecen **mezclas** del ruido y la curva
final, cada vez con menos ruido. Es el camino recto de L6.2
(`x_t = t · ruido + (1 − t) · bloque`), ahora recorrido de verdad.

El resultado se parece a lo que hizo la persona, con errores: el codo se queda
unos 4° corto y la muñeca sube cuando la persona la bajó. En este instante, el
error promedio en las 26 articulaciones es 1,8°.

## 3. Varios sorteos para la misma escena

Un VLA con flow matching no da **una** respuesta: sortea. Mismo instante,
mismo modelo, 20 ruidos distintos:

In [ ]:
samples = []
with torch.no_grad():
    for seed in range(20):
        torch.manual_seed(seed)
        n = torch.randn_like(noise)
        samples.append(to_radians(vla.sample_with_path(model, obs, n)[-1])[0])
samples = np.stack(samples)  # (20, 50, 26)

fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
for ax, j in zip(axes, watch):
    for s in samples:
        ax.plot(seconds, s[:, j], color="tab:red", alpha=0.3, lw=1)
    ax.plot(seconds, truth[:, j], color="black", ls="--", label="la persona")
    ax.set_title(joint_names[j], fontsize=9)
    ax.set_xlabel("s")
axes[0].legend(fontsize=8)
plt.tight_layout()
spread = np.degrees(samples.std(axis=0).mean(axis=0))
print("cuánto difieren los sorteos (desvío medio, grados):")
for j in np.argsort(-spread)[:5]:
    print(f"  {joint_names[j]:<22} {spread[j]:.1f}°")
print(f"  ... promedio de las 26: {spread.mean():.1f}°")

Los 20 sorteos salen casi iguales: difieren 0,3° en promedio (0,5° en el
codo, lo que más varía). Y se equivocan **juntos**: los 20 suben la muñeca
cuando la persona la bajó. La diferencia entre sorteos es mucho más chica que
la distancia a la persona.

Compará con el árbol de L6: ahí había dos maneras de hacer la tarea (por
izquierda o por derecha) y los sorteos se repartían entre las dos. Acá el
modelo ve una sola manera, y tiene sentido: nuestras demostraciones son una
persona haciendo la misma tarea de la misma forma. Con estos datos, entonces,
el error no viene del sorteo: viene de que el modelo se equivoca siempre para
el mismo lado.

Un detalle en el pulgar: el de la persona va en escalones (la mano se comanda
de a saltos) y el modelo lo convierte en una rampa suave.

## 4. ¿Cuántas pasadas hacen falta?

Mismo ruido, distinta cantidad de pasadas. Medimos el error contra la persona y
cuán "temblequeante" sale el bloque (cuánto cambia la velocidad de una acción a
la siguiente; más bajo es más suave).

In [ ]:
def roughness(block):
    return np.degrees(np.abs(np.diff(block, n=2, axis=0)).mean())


rows = []
with torch.no_grad():
    for n_steps in [1, 2, 3, 5, 10, 20]:
        if device == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        block = to_radians(vla.sample_with_path(model, obs, noise, num_steps=n_steps)[-1])[0]
        if device == "cuda":
            torch.cuda.synchronize()
        rows.append((n_steps, np.degrees(np.abs(block - truth).mean()), roughness(block), 1000 * (time.perf_counter() - start)))
        if n_steps in (1, 10):
            globals()[f"block_{n_steps}"] = block

print(f"{'pasadas':>8} {'error (°)':>10} {'temblor (°)':>12} {'tiempo (ms)':>12}")
for n_steps, error, rough, ms in rows:
    print(f"{n_steps:>8} {error:>10.1f} {rough:>12.2f} {ms:>12.0f}")
print(f"{'persona':>8} {'':>10} {roughness(truth):>12.2f}")

fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
for ax, j in zip(axes, watch):
    ax.plot(seconds, truth[:, j], color="black", ls="--", label="la persona")
    ax.plot(seconds, block_1[:, j], color="tab:blue", label="1 pasada")
    ax.plot(seconds, block_10[:, j], color="tab:red", label="10 pasadas")
    ax.set_title(joint_names[j], fontsize=9)
    ax.set_xlabel("s")
axes[0].legend(fontsize=8)
plt.tight_layout()

El error casi no cambia: 1,9° con una pasada y 1,8° de dos en adelante. Lo
que cambia es el **temblor**: con una pasada la curva azul sigue la forma pero
tiembla (1,12°); con 10 baja a 0,24° y con 20 a 0,19°. La persona: 0,12°.

¿Por qué una sola pasada ya acierta, si en L6.1 daba el promedio? Porque acá,
como vimos en la sección 3, hay una sola manera de hacer la tarea: el promedio
*es* la respuesta.

¿Y por qué tiembla? Con una pasada, el modelo tiene que sacarle al bloque, de
un saque, exactamente el ruido que se sorteó. Lo que no logra sacar queda en la
curva como temblor, y cada pasada extra corrige parte de ese resto.

El costo: la fila del VLM se calcula una vez (unos 20 a 30 ms) y cada pasada
suma unos 10 ms. Con 10 pasadas son unos 130 ms (varía un poco entre
corridas), que sobran para un bloque que dura 1,7 s. Con 5 pasadas tarda la mitad y tiembla un poco más (0,33°).

## Lo que nos llevamos

- Generar un bloque es la caminata de L6.3 en 1600 dimensiones: 10 pasadas
  desde el ruido.
- Distintos ruidos dan bloques distintos para la misma escena: el modelo
  sortea, y cuánto varía depende de la articulación.
- Con una sola pasada vuelve a aparecer el promedio de L6.1; con pocas pasadas
  ya se acerca al resultado final.

La pregunta que queda es si lo que sortea **está bien**: en `07` lo medimos en
los 7 episodios que nunca vio, con la misma vara que a la primera ACT.

## Referencias

- Shukor et al., [SmolVLA](https://arxiv.org/abs/2506.01844) (2025).
- Lipman et al., [Flow Matching Guide and Code](https://arxiv.org/abs/2412.06264)
  (2024).